# 第8回 深層学習2：再構成・記憶・生成

目標：数値を予想し、計算を実行し、一つの条件を変えて理由を説明する。第6・7回の重み・損失・RNNの状態を使う。

このNotebookだけで実行でき、同梱Pythonのimport・外部データ・GPUは不要。Python 3.10以降の既存venvをカーネルに選び、**再起動して全セル実行**する。計算は標準ライブラリのみ。Jupyterカーネルがなければ既存venvで `python -m pip install ipykernel`。詳細は同梱のHTML解説を読む。

元のNotebookを残し、コピーで実験する。予想はMarkdownセルや自分のメモへ記録する。

## 1. 自己符号化器の準備と処理

5行の人工入力（1の位置だけが異なる）を5→3→5のネットワークで再構成する。目標は入力自身。全25出力の平均二乗誤差を使う。シード65535、学習率4、1,000回。

勾配を全て計算してから重みを更新する。学習結果はこの5行での適合であり、未知データの性能を示さない。

In [ ]:
"""第8回：再構成・記憶・生成を小さな数値で観察する。標準ライブラリのみ。"""
import argparse
import copy
import json
import math

SEED = 65535
DATA = [[float(i == j) for j in range(5)] for i in range(5)]
REAL = [1.0, 2.0, 3.0]
NOISE = [-1.0, 0.0, 1.0]
GAN_INITIAL = {"a": 0.5, "b": -1.0, "w": 1.0, "d": 0.0}


def sigmoid(x):
    if x >= 0:
        return 1 / (1 + math.exp(-x))
    e = math.exp(x)
    return e / (1 + e)


def initial_autoencoder(seed=SEED):
    """JSでも再現できる32bit LCG。教材用で暗号用途ではない。"""
    state = seed
    def value():
        nonlocal state
        state = (1664525 * state + 1013904223) % (2 ** 32)
        return 2 * state / (2 ** 32) - 1
    return {"enc": [[value() for _ in range(6)] for _ in range(3)],
            "dec": [[value() for _ in range(4)] for _ in range(5)]}


def reconstruct(x, weights):
    if len(x) != 5:
        raise ValueError("入力は5個の数値です")
    hidden = [sigmoid(sum(w[j] * x[j] for j in range(5)) + w[5])
              for w in weights["enc"]]
    output = [sigmoid(sum(w[j] * hidden[j] for j in range(3)) + w[3])
              for w in weights["dec"]]
    return {"hidden": hidden, "output": output,
            "mse": sum((output[j] - x[j]) ** 2 for j in range(5)) / 5}


def ae_loss(weights, data=DATA):
    return sum(reconstruct(x, weights)["mse"] for x in data) / len(data)


def ae_gradient(weights, data=DATA):
    grad = {k: [[0.0 for _ in row] for row in rows] for k, rows in weights.items()}
    for x in data:
        r = reconstruct(x, weights)
        h, y = r["hidden"], r["output"]
        dy = [2 * (y[k] - x[k]) * y[k] * (1 - y[k]) / (5 * len(data))
              for k in range(5)]
        # 出力側も中間層側も、更新前の同じ重みから勾配を計算する。
        dh = [sum(dy[k] * weights["dec"][k][j] for k in range(5)) * h[j] * (1 - h[j])
              for j in range(3)]
        for k in range(5):
            for j in range(4):
                grad["dec"][k][j] += dy[k] * (h[j] if j < 3 else 1)
        for j in range(3):
            for i in range(6):
                grad["enc"][j][i] += dh[j] * (x[i] if i < 5 else 1)
    return grad


def train_autoencoder(steps=1000, rate=4.0, data=DATA, initial=None):
    if not isinstance(steps, int) or not 0 <= steps <= 4000 or not 0 <= rate <= 4:
        raise ValueError("学習回数は0〜4000の整数、学習率は0〜4です")
    weights = copy.deepcopy(initial if initial is not None else initial_autoencoder())
    history = [{"step": 0, "mse": ae_loss(weights, data)}]
    for step in range(1, steps + 1):
        grad = ae_gradient(weights, data)
        for key, rows in weights.items():
            for i, row in enumerate(rows):
                for j in range(len(row)):
                    row[j] -= rate * grad[key][i][j]
        if step % 100 == 0 or step == steps:
            history.append({"step": step, "mse": ae_loss(weights, data)})
    return {"weights": weights, "history": history,
            "reconstructions": [reconstruct(x, weights) for x in data]}



## 2. LSTMの処理

手動ゲートは `c=f×前のc+i×g`、`h=o×tanh(c)`。前のc=1、f=0.8、i=0.5、g=0.6、o=0.75なら、残す量0.8と書く量0.3からc=1.1、h≈0.600374。

`memory_sequence` は初期c=0、i=o=1、候補[1,0,0,0]の理想化例。候補は生の入力とは異なる。`fixed_lstm_step` は入力と前の出力からゲートを求めるが、重みは説明用の固定値で未学習。

In [ ]:
def gate_step(previous=1.0, forget=0.8, input_gate=0.5, candidate=0.6, output_gate=0.75):
    """ゲートを手動指定したセル計算。ゲートの学習実験ではない。"""
    if not all(0 <= v <= 1 for v in (forget, input_gate, output_gate)) or not -1 <= candidate <= 1:
        raise ValueError("ゲートは0〜1、候補は-1〜1です")
    kept = forget * previous
    written = input_gate * candidate
    cell = kept + written
    return {"kept": kept, "written": written, "cell": cell,
            "hidden": output_gate * math.tanh(cell)}


def fixed_lstm_step(x, previous_cell=0.0, previous_hidden=0.0):
    """固定した重みからゲートを計算する1次元LSTM（未学習）。"""
    f = sigmoid(0.5 * x + 0.2 * previous_hidden + 1)
    i = sigmoid(x - 0.1 * previous_hidden)
    g = math.tanh(x + 0.3 * previous_hidden)
    o = sigmoid(0.4 * x + 0.1 * previous_hidden)
    return dict(gate_step(previous_cell, f, i, g, o), forget=f, input_gate=i,
                candidate=g, output_gate=o)


def memory_sequence(candidates=(1.0, 0.0, 0.0, 0.0), forget=0.8):
    """入力・出力ゲートを1とした理想化例。候補値は生の入力とは区別する。"""
    cell = 0.0
    result = []
    for candidate in candidates:
        step = gate_step(cell, forget, 1, candidate, 1)
        cell = step["cell"]
        result.append(step)
    return result



## 3. GANの処理

G(z)=az+b、D(x)=sigmoid(wx+d)。実データ[1,2,3]、ノイズ[-1,0,1]、初期(a,b,w,d)=(0.5,-1,1,0)、学習率0.1。

Dは実データを1、生成データを0へ近づける交差エントロピーを減らす。Gは平均−log D(G(z))を減らす。D更新時はGを固定、G更新時はDを固定する。学習成功を保証しない小さな一巡の実験。

In [ ]:
def generate(z, params):
    return params["a"] * z + params["b"]


def discriminate(x, params):
    return sigmoid(params["w"] * x + params["d"])


def softplus(x):
    return max(x, 0) + math.log1p(math.exp(-abs(x)))


def gan_losses(params):
    real_logits = [params["w"] * x + params["d"] for x in REAL]
    fake_logits = [params["w"] * generate(z, params) + params["d"] for z in NOISE]
    return {"d_loss": sum(softplus(-s) for s in real_logits) / len(REAL)
            + sum(softplus(s) for s in fake_logits) / len(NOISE),
            "g_loss": sum(softplus(-s) for s in fake_logits) / len(NOISE)}


def gan_gradient(params, target):
    if target == "d":
        grad = {"w": 0.0, "d": 0.0}
        for xs, label in ((REAL, 1), ([generate(z, params) for z in NOISE], 0)):
            for x in xs:
                delta = (discriminate(x, params) - label) / len(xs)
                grad["w"] += delta * x
                grad["d"] += delta
        return grad
    if target == "g":
        grad = {"a": 0.0, "b": 0.0}
        for z in NOISE:
            delta = (discriminate(generate(z, params), params) - 1) * params["w"] / len(NOISE)
            grad["a"] += delta * z
            grad["b"] += delta
        return grad
    raise ValueError("更新対象はdまたはgです")


def gan_step(params, target, rate=0.1):
    if not 0 <= rate <= 1:
        raise ValueError("学習率は0〜1です")
    updated = dict(params)
    for key, value in gan_gradient(params, target).items():
        updated[key] -= rate * value
    return updated



## 4. 基準結果

まず下のセルを実行する前に、LSTMのセル状態を手計算する。実行後は自己符号化器MSE 0.316058→0.004639、gatesのc=1.1、h=0.600374を確認する。

GANのD更新後はa,bが不変、G更新後はw,dが不変であることを表す辞書を読む。値は表示用に6桁へ丸める。

In [ ]:
def rounded(value):
    if isinstance(value, float):
        return round(value, 6)
    if isinstance(value, list):
        return [rounded(v) for v in value]
    if isinstance(value, dict):
        return {k: rounded(v) for k, v in value.items()}
    return value


def demo():
    trained = train_autoencoder()
    after_d = gan_step(GAN_INITIAL, "d")
    after_g = gan_step(after_d, "g")
    return rounded({
        "autoencoder": {"before": ae_loss(initial_autoencoder()),
                        "after": trained["history"][-1]["mse"],
                        "first": trained["reconstructions"][0]},
        "gates": gate_step(), "sequence": memory_sequence(),
        "fixed_lstm": fixed_lstm_step(1),
        "gan": {"initial": dict(GAN_INITIAL), "after_d": after_d, "after_g": after_g,
                "generated": [generate(z, after_g) for z in NOISE], "losses": gan_losses(after_g)}})


baseline_result = demo()
print(json.dumps(baseline_result, ensure_ascii=False, indent=2))

## 変更実験1：入力

予想：1の位置を変えると、中間表現と再構成はどう変わるか。固定初期重みで入力だけを比較する。リストのコピーを使うので元のデータ5行は変わらない。

In [ ]:
trial_x = DATA[1][:]
trial_reconstruction = reconstruct(trial_x, initial_autoencoder())
print("入力:", trial_x, "再構成:", rounded(trial_reconstruction))

## 変更実験2：学習回数

予想：同じ初期値から2,000回学習したらMSEはどうなるか。基準の1,000回とは別に実行する。想定結果は約0.001924。回数以外の条件は同じか確認する。

In [ ]:
trial_fit = train_autoencoder(2000)
print("2000回のMSE:", round(trial_fit["history"][-1]["mse"], 6))

## 変更実験3：出力ゲート

予想：o=0ならcも0になるか。想定結果はc=1.1、h=0。出力の抑制と忘却の違いを説明する。

In [ ]:
trial_gate = gate_step(output_gate=0)
print(rounded(trial_gate))

## 変更実験4：順序

予想：[1,0,0,0]を逆順にしたら最後の状態はどうなるか。コピーを逆順にし、元のリストを残す。最後のcは1となる。書き込んだ後に何回fを掛けたかを説明する。

In [ ]:
baseline_candidates = [1.0, 0.0, 0.0, 0.0]
trial_candidates = baseline_candidates[::-1]
trial_sequence = memory_sequence(trial_candidates)
print("元:", baseline_candidates, "変更後:", trial_candidates)
print(rounded(trial_sequence))

## 変更実験5：生成器のバイアス

予想：bを−1から0へ変えると、z=0の生成値と判定はどうなるか。想定はG(0)=0、D(G(0))=0.5。手動変更と学習を区別する。

In [ ]:
trial_params = dict(GAN_INITIAL)
trial_params["b"] = 0
trial_generated = generate(0, trial_params)
trial_decision = discriminate(trial_generated, trial_params)
print("生成値:", trial_generated, "判定:", trial_decision)

## 変更実験6：学習率を0へ

予想：GANで学習率が0ならどのパラメーターが変わるか。コピーからD→Gを更新する。全て初期値のままであることを確認する。

In [ ]:
trial_zero = gan_step(dict(GAN_INITIAL), "d", rate=0)
trial_zero = gan_step(trial_zero, "g", rate=0)
print("初期:", GAN_INITIAL, "変更後:", trial_zero)

## 結果を説明する・元に戻す

各実験で「変えた値」「固定した条件」「実際の出力」「その理由」を記録する。基準結果は `baseline_result` に残してある。

NameErrorなら再起動して全セル実行。予想と違うときは一つずつ条件を確認する。編集したコードを戻すには元のZIPから新しいフォルダーへ展開する。カーネルの再起動だけではソースの編集は戻らない。

Python版はvenv有効化後に `python deep_learning2.py --demo`。uvを使う場合は `uv run --no-project python deep_learning2.py --demo`。

## 参考資料

- [Deep Learning 第14章](https://www.deeplearningbook.org/contents/autoencoders.html)
- [PyTorch LSTMの式](https://docs.pytorch.org/docs/2.14/generated/torch.nn.LSTM.html)
- [Generative Adversarial Nets](https://papers.nips.cc/paper/2014/hash/f033ed80deb0234979a61f95710dbe25-Abstract.html)

外部の参考資料の閲覧だけはネット接続が必要。今回の数値例は授業用の新作で、原論文の実験の再現ではない。